In [6]:
from langchain_community.utilities import SQLDatabase

In [7]:
db=SQLDatabase.from_uri("sqlite:///mydb.db")

In [8]:
db

In [9]:
db.dialect
#dialect is the type of database, in this case sqlite

'sqlite'

In [10]:
db.get_usable_table_names()

['customers', 'employees', 'orders']

In [11]:
from langchain_groq import ChatGroq

In [12]:
# Option A: Set it as an environment variable (Cleanest way)
#os.environ["GROQ_API_KEY"] = "your_actual_key_here"

In [13]:
llm=ChatGroq(model="llama-3.1-8b-instant")

#else use a different model, e.g. "gpt-4o"
#llm = ChatGroq(model="llama-3.3-70b-versatile")

In [14]:
llm.invoke("hello how are you?")

AIMessage(content="Hello. I'm just a language model, I don't have emotions or feelings like humans do, so I don't have a physical state of being. However, I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 40, 'total_tokens': 99, 'completion_time': 0.06577593, 'completion_tokens_details': None, 'prompt_time': 0.003069417, 'prompt_tokens_details': None, 'queue_time': 0.068346851, 'total_time': 0.068845347}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1a67-0b9a-7c01-9d42-f64f96599ef3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 59, 'total_tokens': 99})

In [15]:
#SO LLM IS WORKING

In [16]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [17]:
toolkit=SQLDatabaseToolkit(db=db, llm=llm)

In [18]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>),
 QuerySQLCheckerTool(description='Use this tool to 

In [19]:

tools=toolkit.get_tools()
tools

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>),
 QuerySQLCheckerTool(description='Use this tool to 

In [20]:
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [21]:
list_tables_tool = next((tool for tool in tools if tool.name == "sql_db_list_tables"), None)

In [22]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>)

In [23]:
get_schema_tool = next((tool for tool in tools if tool.name == "sql_db_schema"),None)

In [24]:
get_schema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000002478146B590>)

In [25]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [26]:
get_schema_tool.invoke("customers")

'\nCREATE TABLE customers (\n\tcustomer_id INTEGER, \n\tfirst_name TEXT NOT NULL, \n\tlast_name TEXT NOT NULL, \n\temail TEXT NOT NULL, \n\tphone TEXT, \n\tPRIMARY KEY (customer_id), \n\tUNIQUE (email)\n)\n\n/*\n3 rows from customers table:\ncustomer_id\tfirst_name\tlast_name\temail\tphone\n1\tArjun\tMehta\tarjun.m@example.com\t9876543210\n2\tSara\tKhan\tsara.k@gmail.com\t9821098765\n3\tVikram\tIyer\tviyer@outlook.com\t9900887766\n*/'

In [27]:
#instead of this, we can print it cleanly
print(get_schema_tool.invoke("customers"))


CREATE TABLE customers (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
customer_id	first_name	last_name	email	phone
1	Arjun	Mehta	arjun.m@example.com	9876543210
2	Sara	Khan	sara.k@gmail.com	9821098765
3	Vikram	Iyer	viyer@outlook.com	9900887766
*/


In [28]:
#since it was not possible to connect the database directly to the langchain workflow, we used the "Langchain wrapper" : "`SQLDatabaseToolkit` to connect the database to the LLM, and then we used the tools provided by the toolkit to interact with the database."

## Custom tool creation


In [33]:
from langchain_core.tools import tool

@tool    
#this is a tool decorator that convert any sort of a fn into a tool that can be used by the agent
def db_query_tool(query:str)->str:
    """
    Execute a SQL query against the database and return the results as a string.
    If the query fails, return an error message instead of throwing an exception.
    In case of an error, the function will return a string with the error message instead of throwing an exception.
    """
    #this is used to show description of a tool!
    
    result=db.run_no_throw(query)
    #no_throw means that if the query fails, it will not throw an error, but it will return None

    if not result:
        return "Error: Query failed. Please check your query and try again."
    return result

In [34]:
db_query_tool.invoke("SELECT * FROM customers;")

"[(1, 'Arjun', 'Mehta', 'arjun.m@example.com', '9876543210'), (2, 'Sara', 'Khan', 'sara.k@gmail.com', '9821098765'), (3, 'Vikram', 'Iyer', 'viyer@outlook.com', '9900887766'), (4, 'Ananya', 'Das', 'ananya_das@yahoo.com', '9123456789'), (5, 'Rohan', 'Gupta', 'rohan.g@techcorp.in', '9887766554')]"

In [35]:
#we also have run() method, it will gen same result but if the query fails, it will throw an exception instead of returning an error message.


# Prompt template


### query_check and query_gen

In [37]:
from langchain_core.prompts import ChatPromptTemplate

In [38]:
query_check_system = """You are a SQL expert with a strong attention to detail.
Double check the SQLite query for common mistakes, including:
- Using NOT IN with NULL values
- Using UNION when UNION ALL should have been used
- Using BETWEEN for exclusive ranges
- Data type mismatch in predicates
- Properly quoting identifiers
- Using the correct number of arguments for functions
- Casting to the correct data type
- Using the proper columns for joins

If there are any of the above mistakes, rewrite the query. If there are no mistakes, just reproduce the original query.

You will call the appropriate tool to execute the query after running this check.
"""

query_check_prompt = ChatPromptTemplate.from_messages([("system", query_check_system), ("placeholder", "{messages}")])

In [39]:
query_check = query_check_prompt | llm.bind_tools([db_query_tool])
#the above code is creating a new tool called "query_check" that takes a SQL query as input, checks it for common mistakes using the LLM, and then executes it using the "db_query_tool" if it passes the check.   
#the llm.bind_tools() method is used to bind the "db_query_tool" to the LLM, so that it can be called from within the LLM's response generation process. This allows the LLM to call the "db_query_tool" to execute the SQL query after checking it for mistakes.


query_check.invoke({"messages": [("user", "SELECT * FROM customers;")]})
#this will check the query for mistakes and then execute it using the db_query_tool if it passes the check.

AIMessage(content='SELECT * FROM customers;', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 439, 'total_tokens': 445, 'completion_time': 0.012805684, 'completion_tokens_details': None, 'prompt_time': 0.026521071, 'prompt_tokens_details': None, 'queue_time': 0.051908749, 'total_time': 0.039326755}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1a9f-8a10-7bf1-b2bc-dd270068590c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 439, 'output_tokens': 6, 'total_tokens': 445})